# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from mlxtend.frequent_patterns import apriori
# 学号: 0234972

#制定画图风格
plt.style.use('fivethirtyeight') 
#将警告过滤掉
warnings.filterwarnings('ignore')

# ======================================
# ## 🍎 数据导入与初探
# ======================================

# 导入CSV数据并初步查看 (作为备份参考，实际分析使用交易表.txt)
# data_csv = pd.read_csv('E:\Demo\python\超市商品购买关联分析\data\数据-某日顾客商品购买明细.sav', encoding='GBK')
# print('======================================')
# print('CSV数据前5行预览:')
# print(data_csv.head(5))

# print('======================================')
# print('CSV数据信息:')
# data_csv.info()

# 导入TXT交易数据并处理
# 假设 '顾客交易表.txt' 和 '某日顾客商品购买明细.txt' 内容相同或可互换，
# 我们使用后者的名称来保持与您原始代码的关联，并简化流程。
try:
    with open(r'顾客交易表.txt', encoding='utf-8') as f:
        df = pd.read_table(f, sep=r'\s*,\s*', engine='python', header=None, names=["会员ID", "商品", "数量"])
except FileNotFoundError:
    # 如果文件路径或名称有差异，可以尝试另一个文件名或路径
    print("Warning: '顾客交易表.txt' not found, attempting to read '某日顾客商品购买明细.txt'...")
    try:
        with open(r'某日顾客商品购买明细.txt', encoding='utf-8') as f:
            # 假设文件内容是 '会员ID,商品,数量' 格式
            df = pd.read_table(f, sep=r'\s*,\s*', engine='python', header=None, names=["会员ID", "商品", "数量"])
    except Exception as e:
        print(f"Error reading data file: {e}")
        # 如果读取失败，请检查文件路径和内容格式
        raise

print('======================================')
print('处理后的交易数据前5行预览:')
print(df.head(5))

print('======================================')
print('数据缺失值统计:')
# 检查处理后的 DataFrame 的缺失值
print(df.isnull().sum())
# ⚠️ 注意: 您的原代码中对 df.describe() 的调用对非数值列意义不大，在此省略，或替换为更具意义的统计（如商品种类数）

# ======================================
# ## 📊 数据转换：构建One-Hot编码交易矩阵
# ======================================

# 使用数据透视表(pivot_table)将交易明细转化为：
# 索引(index)为 会员ID，列(columns)为 商品，值为 数量
df_v = df.pivot_table(index='会员ID', columns='商品', values='数量', aggfunc='sum')

# 将数量矩阵转换为**二元交易矩阵 (One-Hot Encoding)**：
# 如果会员购买了某商品 (数量 > 0 或非 NaN)，则标记为 1 (True)，否则标记为 0 (False)。
# 使用 pd.isna(x) 判断是否为未购买 (NaN)，然后用 applymap 转换为 0 或 1。
# 在使用 aggfunc='sum' 后，NaN 表示未购买，非 NaN 表示已购买 (数量 >= 1)
data = df_v.applymap(lambda x: 0 if pd.isna(x) or x == 0 else 1) 

print('======================================')
print('One-Hot编码交易矩阵前5行预览 (1=购买, 0=未购买):')
print(data.head())


# ======================================
# ## 📈 关联规则挖掘：Apriori算法
# ======================================

# 利用 Apriori 找出**频繁项集**
# 设置最小支持度 min_support=0.1，并使用列名 use_colnames=True
'''
引入函数 apriori(df, min_support=0.5, use_colnames=False, max_len=None, verbose=0, low_memory=False)
参数说明：
    df：数据集（通常是 One-Hot 编码的交易矩阵）
    min_support：给定的最小支持度（这里设置为 0.1）。
    use_colnames：为 True 时，返回的物品组合直接显示物品名称，而不是编号。
'''
freq = apriori(data, min_support=0.1, use_colnames=True)

print('======================================')
print(f'🥳 频繁项集计算结果 (最小支持度 min_support=0.1): 您的学号是0234972')
# 打印结果，展示支持度 (support) 和商品项集 (itemsets)
print(freq)

# 可以进一步计算关联规则 (如 Lift/Confidence)，这里只执行到频繁项集，以匹配您的原始代码要求。
# from mlxtend.frequent_patterns import association_rules
# rules = association_rules(freq, metric="lift", min_threshold=1.0)
# print('======================================')
# print('关联规则结果:')
# print(rules.head())
